# DP2 —  Source catalog query over a DDF via Butler

**Author:** dagoret  
**Creation Date:** 2026-03-17  
**Last Date:** 2026-03-17  
**version:** v0
## Purpose

Retrieve **Object**, **Source**, and **ForcedSource** catalogs
for a user-selected Deep Drilling Field (DDF) using **Butler Gen3 only**
(no TAP service, not yet available for DP2 at USDF).

## Actual schema (discovered at runtime — COSMOS, LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545)

- `dia_object` : dims `(skymap, tract)` — 1 ref per tract, index = `diaObjectId`
- `dia_source` : dims `(skymap, tract)` — 1 ref per tract, index = `diaSourceId` (join on `diaObjectId` column)
- `dia_object_forced_source` : dims `(skymap, tract, patch)` — **21 refs per tract** (one per patch)
  - Must iterate over all patch refs and concat
  - Contains per-visit forced photometry linked to DiaObjects

## Reference notebooks
- `2026-03-07_AccessToDP2.ipynb` — Butler setup
- `2026-03-13_DP2_DDF_Tracts_SkyMap_Mollweide.ipynb` — tract lookup

---
## 0. Imports

In [ ]:
import warnings

warnings.filterwarnings("ignore")
import traceback
import gc


import os

import numpy as np
import pandas as pd
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from astropy.coordinates import SkyCoord
from astropy.table import Table
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import Angle

from scipy.stats import median_abs_deviation

from lsst.daf.butler import Butler
from lsst.geom import SpherePoint, degrees

print(f"Matplotlib : {matplotlib.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("Imports OK")

In [ ]:
mpl.rcParams.update(
    {
        "figure.figsize": (8, 5),  # taille par défaut des figures
        "font.size": 14,  # taille de base
        "axes.titlesize": 18,  # titre de l'axe
        "axes.labelsize": 16,  # labels x et y
        "xtick.labelsize": 14,  # ticks x
        "ytick.labelsize": 14,  # ticks y
        "legend.fontsize": 14,  # texte de la légende
        "legend.title_fontsize": 15,  # titre de la légende
        "figure.titlesize": 20,  # titre global de la figure
    }
)

In [ ]:
# Remove to run faster the notebook
# import ipywidgets as widgets
# %matplotlib widget

# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
# try:
#    import ipympl  # noqa: F401
#    %matplotlib widget
#    print("ipympl found → interactive backend (%matplotlib widget)")
# except ImportError:
#    %matplotlib inline
#    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
#    print("Install with:  pip install ipympl")


In [ ]:
# output dirs
NB_TAG = "DP2_DDF_STATICOBJ_BUTLER_01"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")


# Output directory for DRP data
OUTPUT_DIR = "data_DP2_DDF_OBJECT_DRP_01"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Visits filename
FILENAME_VISITS = "visitTable-2025041500138-2026010800515_N34276.csv"

In [ ]:
# debug flags
DEBUG = False
DEBUG_VISITS = False

In [ ]:
# FLAGS
FLAG_FETCH_VISITSEXPOSURES = False

In [ ]:
# =========================================================================
# Helper: add a top x-axis showing calendar dates above the MJD axis
# =========================================================================
# The bottom x-axis of each light curve panel uses MJD.
# This helper adds a twin axis on top with evenly-spaced date labels
# (UTC) so you can immediately read off known observing dates.
#
# Usage (call AFTER plotting, BEFORE tight_layout / savefig):
#   add_date_top_axis(axes[0], n_ticks=6)
# =========================================================================


def add_date_top_axis(ax, n_ticks=6, date_fmt="%Y-%m-%d"):
    """
    Add a secondary x-axis on top of `ax` showing calendar dates.

    The secondary axis shares the same MJD limits as the primary axis
    but labels ticks as UTC calendar dates (e.g. 2025-06-15).

    Parameters
    ----------
    ax : matplotlib Axes
        Primary axes whose bottom x-axis carries MJD values.
    n_ticks : int
        Number of evenly-spaced tick marks on the top axis.
    date_fmt : str
        strftime format for the date labels (default YYYY-MM-DD).

    Returns
    -------
    ax_top : matplotlib Axes
        The new twin axes placed on top.
    """
    mjd_min, mjd_max = ax.get_xlim()

    # Build evenly-spaced MJD tick positions spanning the current x range
    mjd_ticks = np.linspace(mjd_min, mjd_max, n_ticks)

    # Convert each MJD tick to a calendar date string via astropy
    date_labels = [Time(m, format="mjd", scale="utc").to_value("isot")[:10] for m in mjd_ticks]

    # Create a twin axis that shares the same x-scale
    ax_top = ax.twiny()
    ax_top.set_xlim(mjd_min, mjd_max)  # must match primary axis exactly
    ax_top.set_xticks(mjd_ticks)
    ax_top.set_xticklabels(date_labels, rotation=30, ha="left", fontsize=8)
    ax_top.set_xlabel("Date (UTC)", fontsize=9, labelpad=4)

    return ax_top


print("add_date_top_axis() helper defined.")

In [ ]:
def load_visits_file(file_path):
    """
    Checks if the visits file exists and loads it into a DataFrame.
    """
    if not os.path.exists(file_path):
        print(f"ERROR: File not found at {file_path}")
        return None

    try:
        # Standard LSST tables are usually Parquet
        if file_path.endswith(".parquet"):
            df_visits = pd.read_parquet(file_path)
        else:
            df_visits = pd.read_csv(file_path)

        print(f"Successfully loaded {len(df_visits):,} visits from {os.path.basename(file_path)}")
        return df_visits
    except Exception as e:
        print(f"ERROR loading file: {e}")
        return None


# --- Usage ---
# df_visits = load_visits_file(FULLFILENAME_VISITS)

---
## 1. User Configuration

In [ ]:
SELECTED_DDF = "COSMOS"  # COSMOS | ECDFS | ELAIS-S1 | XMM-LSS | EDFS-a | EDFS-b | EDFS | M49
# Check the selected TRACT and PATCH exist for the selected DDF
SELECTED_TRACT = 9813
SELECTED_PATCH = 83
CONE_RADIUS_DEG = 1.8
MIN_NSOURCES = 500

REPO = "dp2_prep"
COLLECTION = "LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545"
SKYMAP_NAME = "lsst_cells_v2"
INSTRUMENT = "LSSTCam"
WHERE_CLAUSE_INSTRUMENT = "instrument = '" + f"{INSTRUMENT}" + "'"
DATE_START = 20250415
WHERE_CLAUSE_DATE = WHERE_CLAUSE_INSTRUMENT + f" and day_obs >= {DATE_START}"

TRACT_SEARCH_RADIUS_DEG = 1.8

DDF_COORDS = {
    "COSMOS": (150.119, +2.206),
    "ECDFS": (53.125, -28.100),
    "ELAIS-S1": (9.450, -44.000),
    "XMM-LSS": (35.708, -4.750),
    "EDFS-a": (58.900, -49.315),
    "EDFS-b": (63.600, -47.600),
    "EDFS": (61.240, -48.423),
    "M49": (187.400, +8.000),
}

RA_CENTER, DEC_CENTER = DDF_COORDS[SELECTED_DDF]
print(f"DDF         : {SELECTED_DDF}  ({RA_CENTER:.4f}°, {DEC_CENTER:+.4f}°)")
print(f"Cone radius : {CONE_RADIUS_DEG}°")
print(f"Min nDiaSrc : {MIN_NSOURCES}")
print(f"Collection  : {COLLECTION}")
print(f"Instrument  : {INSTRUMENT}")
print(f"Where clause  : {WHERE_CLAUSE_INSTRUMENT}")
print(f"Where clause date : {WHERE_CLAUSE_DATE}")

---
## 2. Butler and SkyMap

In [ ]:
butler = Butler(REPO, collections=COLLECTION)
registry = butler.registry
print("Butler OK")

In [ ]:
try:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME, collections=COLLECTION)
except Exception:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME)
print(f"SkyMap '{SKYMAP_NAME}' : {len(skymap)} tracts")

### 2.1 Search for visit tables

In [ ]:
visitTable_pattern1 = "*isitTable*"
visitTable_pattern2 = "*isitTable"
visitTable_name = "preVisitTable"

## 2.2 Retrieve the science visits in order to match the visitid with the MJD in forced source photometry
- `visitid = id`
- `MJD = mjd `

In [ ]:
if FLAG_FETCH_VISITSEXPOSURES:
    visitTable = getLostOfVisits(registry, where_clause_date=WHERE_CLAUSE_DATE)
    print(visitTable.head(1))
    visitMin, visitMax, Nvisits = visitTable["id"].min(), visitTable["id"].max(), len(visitTable)
    print(f"visitMin = {visitMin},visitMax = {visitMax}, Nvisits = {Nvisits}")
    visit_filename = f"visitTable-{visitMin}-{visitMax}_N{Nvisits}.csv"
    print(f"filename_visits = {visit_filename}")
    visit_fullfilename = os.path.join(OUTPUT_DIR, visit_filename)
    visitTable.to_csv(visit_fullfilename)
    del visitTable
    gc.collect()

---
## 3. Dataset Type Inventory

Identify the correct names and **dimensions** for DIA products — in particular
`dia_object_forced_source` has `(skymap, tract, patch)` dimensions, which
requires iterating over patch refs.

In [ ]:
# Hard-coded after discovery: confirmed present in DM-54249
DATASET_TYPE_OBJ = "object"  # dims: skymap, tract
DATASET_TYPE_SRC = "source"  # dims: skymap, tract
DATASET_TYPE_FORCED = "object_forced_source"  # dims: skymap, tract, patch


# Print actual dimensions for confirmation
for dt_name in [DATASET_TYPE_OBJ, DATASET_TYPE_SRC, DATASET_TYPE_FORCED]:
    try:
        dt = registry.getDatasetType(dt_name)
        in_col = registry.queryDatasets(dt_name, collections=COLLECTION).any(execute=False, exact=False)
        print(f"{dt_name:40s}  dims={list(dt.dimensions.names)}  present={in_col}")
    except Exception as exc:
        print(f"{dt_name:40s}  ERROR: {exc}")

---
## 4. Find Tracts Covering the DDF

In [ ]:
def find_tracts_for_coord(skymap, ra_deg, dec_deg, radius_deg=1.8):
    cos_dec = max(np.cos(np.deg2rad(dec_deg)), 0.01)
    step = 0.35
    found_ids = set()
    for ddec in np.arange(-radius_deg, radius_deg + step, step):
        for dra in np.arange(-radius_deg, radius_deg + step, step):
            if np.sqrt(dra**2 + ddec**2) > radius_deg:
                continue
            ra_s = ra_deg + dra / cos_dec
            dec_s = dec_deg + ddec
            if not (-89.9 <= dec_s <= 89.9):
                continue
            sp = SpherePoint(ra_s * degrees, dec_s * degrees)
            try:
                found_ids.add(skymap.findTract(sp).tract_id)
            except Exception:
                pass
    return sorted(found_ids)


tract_ids = find_tracts_for_coord(skymap, RA_CENTER, DEC_CENTER, radius_deg=TRACT_SEARCH_RADIUS_DEG)
print(f"Tracts covering {SELECTED_DDF}: {tract_ids}")

---
## 5. Schema Discovery (one-time probe)

### 5.1 Object Schema

In [ ]:
refs_probe = list(
    registry.queryDatasets(
        DATASET_TYPE_OBJ,
        collections=COLLECTION,
        dataId={"skymap": SKYMAP_NAME, "tract": tract_ids[0]},
    )
)
assert refs_probe, "No  object ref found."

obj_raw = butler.get(refs_probe[0])
df_probe = obj_raw.to_pandas() if hasattr(obj_raw, "to_pandas") else obj_raw

print(f"index.name  : {df_probe.index.name!r}")
print(f"index.dtype : {df_probe.index.dtype}")
print(f"index[:3]   : {df_probe.index[:3].tolist()}")
print(f"columns     : {list(df_probe.columns)}")

In [ ]:
# Determine ID column
if df_probe.index.name is not None:
    OBJ_ID_COL = df_probe.index.name
    ID_IN_INDEX = True
    print(f"Object ID in index: '{OBJ_ID_COL}'")
elif any(c in df_probe.columns for c in ["objectId", "object_id"]):
    OBJ_ID_COL = next(c for c in ["objectId", "object_id"] if c in df_probe.columns)
    ID_IN_INDEX = False
    print(f"Object ID as column: '{OBJ_ID_COL}'")
else:
    OBJ_ID_COL = "row_id"
    ID_IN_INDEX = False
    print("WARNING: no ID found, using row index.")

In [ ]:
# Also probe object_forced_source schema (patch-level)
refs_forced_probe = list(
    registry.queryDatasets(
        DATASET_TYPE_FORCED,
        collections=COLLECTION,
        dataId={"skymap": SKYMAP_NAME, "tract": tract_ids[0]},
    )
)
print(f"object_forced_source refs for tract {tract_ids[0]}: {len(refs_forced_probe)}")

if refs_forced_probe:
    frc_raw = butler.get(refs_forced_probe[0])
    df_fprobe = frc_raw.to_pandas() if hasattr(frc_raw, "to_pandas") else frc_raw
    print(f"index.name  : {df_fprobe.index.name!r}")
    print(f"columns     : {list(df_fprobe.columns)}")
    print(f"shape       : {df_fprobe.shape}")

---
## 6. Helper: `to_dataframe()`

In [ ]:
def to_dataframe(obj, id_col=None, id_in_index=None):
    """
    Convert Butler catalog to pandas DataFrame.
    If id_in_index is True, promotes the named index to a regular column.
    Falls back to global OBJ_ID_COL / ID_IN_INDEX if not provided.
    """
    _id_col = id_col if id_col is not None else OBJ_ID_COL
    _id_idx = id_in_index if id_in_index is not None else ID_IN_INDEX

    if isinstance(obj, pd.DataFrame):
        df = obj
    elif hasattr(obj, "to_pandas"):
        df = obj.to_pandas()
    elif hasattr(obj, "to_table"):
        df = obj.to_table().to_pandas()
    elif isinstance(obj, Table):
        df = obj.to_pandas()
    else:
        raise TypeError(f"Cannot convert {type(obj)} to DataFrame.")

    if _id_idx and _id_col not in df.columns:
        df = df.copy()
        df.insert(0, _id_col, df.index)
        df = df.reset_index(drop=True)
    elif _id_col == "row_id" and _id_col not in df.columns:
        df.insert(0, _id_col, range(len(df)))

    return df

### 11.4 Associate Visit and MJD

In [ ]:
FULLFILENAME_VISITS = os.path.join(OUTPUT_DIR, FILENAME_VISITS)
print(f"Visit Filename : {FULLFILENAME_VISITS} ")

In [ ]:
df_visitTable = load_visits_file(FULLFILENAME_VISITS)

In [ ]:
# Create the 'band' column by taking the first character of 'filter'
df_visitTable["band"] = df_visitTable["filter"].str[0]

# Verification: check the unique values
print("Unique bands identified:", df_visitTable["band"].unique())

# Optional: preview the result
display(df_visitTable[["target", "filter", "band"]].head())

In [ ]:
display(df_visitTable.head())

In [ ]:
df_visitTable.target.unique()

In [ ]:
# --- 1. Define the DDF filter ---
# We look for 'y' band and any target containing 'DDF' or 'COSMOS'
is_y_band = df_visitTable["band"] == "y"
is_ddf = df_visitTable["target"].str.contains("DDF|COSMOS|ELAIS|XMM|ECDFS|EDFS", case=False, na=False)

df_y_ddf_visits = df_visitTable[is_y_band & is_ddf].copy()

# --- 2. Identify the unique nights ---
# 'dayObs' is typically the observation date (YYYYMMDD)
nights_y_ddf = sorted(df_y_ddf_visits["day_obs"].unique())

print(f"Found {len(df_y_ddf_visits)} visits in y-band for DDFs.")
print(f"Total nights involved: {len(nights_y_ddf)}")
print(f"First 5 nights: {nights_y_ddf[:]}")

In [ ]:
# 1. Filter for y-band and DDF targets
is_y = df_visitTable["band"] == "y"
is_ddf = df_visitTable["target"].str.contains("DDF|COSMOS|ELAIS|XMM|ECDFS|EDFS", case=False, na=False)

df_y_ddf = df_visitTable[is_y & is_ddf].copy()

# 2. Get the nights and the specific tracts/patches from your forced source data
y_ddf_summary = (
    df_forced_rich.merge(df_y_ddf[["visit", "day_obs", "target"]], on="visit", how="inner")
    .groupby(["target", "day_obs", "tract", "patch"])
    .size()
    .reset_index(name="count")
)

display(y_ddf_summary)